# nnUNetV2 Kidney Stone Segmentation — Training Notebook

**Dataset:** Dataset501_KidneyStones
**Task:** Stone segmentation
**Configuration:** 3D Full Resolution
**Environment:** Google Colab (GPU required)

---

## Instructions
1. Make sure `Dataset501_KidneyStones/` is already inside `MyDrive/1THESIS_AMM/` on Google Drive
2. Run cells sequentially — don't skip
3. After preprocessing, **always run the tar-save cell** before switching runtimes
4. After a runtime reset, run Sections 1–2 then the **tar-restore cell** to recover preprocessed data

## Section 1: Runtime & Environment Setup

### Cell 1.1 — Check GPU

In [ ]:
!nvidia-smi

Tue May 12 13:57:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

### Cell 1.2 — Install nnUNetV2

In [ ]:
!pip install nnunetv2 -q

In [ ]:
import numpy as np
import blosc2

print("NumPy:", np.__version__)
print("Blosc2:", blosc2.__version__)
print("Blosc2 path:", blosc2.__file__)

NumPy: 2.4.4
Blosc2: 4.2.0
Blosc2 path: /usr/local/lib/python3.12/dist-packages/blosc2/__init__.py


### Cell 1.3 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Section 2: Dataset Configuration

### Cell 2.1 — Paths (edit if needed)

In [ ]:
# ============================================================
# Dataset502_KidneyStones — path configuration
# ============================================================

# Where the dataset folder lives on Drive (imagesTr, labelsTr, dataset.json)
DATASET_DRIVE_PATH = "/content/drive/MyDrive/1THESIS_AMM/Dataset502_KidneyStones"

# Where training results (checkpoints) are backed up on Drive
RESULTS_DRIVE_PATH = "/content/drive/MyDrive/1THESIS_AMM/Dataset502_KidneyStones/nnUNet_Results"

# TAR backup path — used to persist preprocessed data across runtime resets
TAR_PREPROCESSED_PATH = TAR_PREPROCESSED_PATH = "/content/drive/MyDrive/1THESIS_AMM/Dataset502_KidneyStones/nnUNet_preprocessed_502.tar"

print("Dataset path  :", DATASET_DRIVE_PATH)
print("Results path  :", RESULTS_DRIVE_PATH)
print("TAR backup    :", TAR_PREPROCESSED_PATH)

Dataset path  : /content/drive/MyDrive/1THESIS_AMM/Dataset502_KidneyStones
Results path  : /content/drive/MyDrive/1THESIS_AMM/Dataset502_KidneyStones/nnUNet_Results
TAR backup    : /content/drive/MyDrive/1THESIS_AMM/Dataset502_KidneyStones/nnUNet_preprocessed_502.tar


### Cell 2.2 — Set nnUNet Environment Variables

In [ ]:
import os

# Raw data lives on Drive (read-only during training)
os.environ["nnUNet_raw"]          = "/content/drive/MyDrive/1THESIS_AMM/Dataset502_KidneyStones/nnUNet_raw"

# Preprocessed data: local SSD (fast I/O for training)
# After preprocessing, this gets tarred to Drive.
# On resume, restore the tar here.
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"

# Results: backed up to Drive after each fold
os.environ["nnUNet_results"]       = "/content/drive/MyDrive/1THESIS_AMM/Dataset502_KidneyStones/nnUNet_results"

for k in ["nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"]:
    print(f"{k}: {os.environ[k]}")

nnUNet_raw: /content/drive/MyDrive/1THESIS_AMM/Dataset502_KidneyStones/nnUNet_raw
nnUNet_preprocessed: /content/nnUNet_preprocessed
nnUNet_results: /content/drive/MyDrive/1THESIS_AMM/Dataset502_KidneyStones/nnUNet_results


### Cell 2.3 — Copy Dataset from Drive → Local SSD

In [ ]:
import shutil
from pathlib import Path

# Create local nnUNet folder structure
!mkdir -p /content/nnUNet_raw /content/nnUNet_preprocessed /content/nnUNet_results

# Copy dataset501 into nnUNet_raw on local SSD
src = Path(DATASET_DRIVE_PATH)
dst = Path("/content/nnUNet_raw/Dataset502_KidneyStones")

if dst.exists():
    print("Dataset already on local SSD — skipping copy.")
else:
    print("Copying dataset from Drive to local SSD...")
    shutil.copytree(src, dst)
    print("Done.")

# Verify
images = list((dst / "imagesTr").glob("*.nii.gz"))
labels = list((dst / "labelsTr").glob("*.nii.gz"))
print(f"Images : {len(images)}")
print(f"Labels : {len(labels)}")
print(f"Match  : {len(images) == len(labels)}")

Copying dataset from Drive to local SSD...
Done.
Images : 290
Labels : 290
Match  : True


## Section 3: Dataset Integrity Check

Quick sanity check before expensive preprocessing.
Reads `dataset.json` and spot-checks 5 random label files.

In [ ]:
import json, random
import numpy as np
import nibabel as nib
from pathlib import Path

base = Path("/content/nnUNet_raw/Dataset502_KidneyStones")

# 1. Load dataset.json
dataset_json = base / "dataset.json"
assert dataset_json.exists(), "dataset.json not found!"
with open(dataset_json) as f:
    meta = json.load(f)
print("Dataset JSON loaded.")
print("Labels       :", meta.get("labels", "NOT FOUND"))
print("Channel names:", meta.get("channel_names", meta.get("modality", "NOT FOUND")))

# Read valid label values from dataset.json (handles any schema)
raw_labels = meta.get("labels", {})
if isinstance(raw_labels, dict):
    valid_label_values = set(int(v) for v in raw_labels.values())
elif isinstance(raw_labels, list):
    valid_label_values = set(range(len(raw_labels)))
else:
    valid_label_values = {0, 1, 2}  # fallback
print("Valid label values:", sorted(valid_label_values))

# 2. Spot-check 5 random labels
lbl_files = list((base / "labelsTr").glob("*.nii.gz"))
sample = random.sample(lbl_files, min(5, len(lbl_files)))

issues = []
for p in sample:
    data = np.asanyarray(nib.load(p).dataobj)
    uniques = set(np.unique(data).astype(int))
    bad = uniques - valid_label_values
    if bad:
        issues.append((p.name, bad))
    print(f"  {p.name}: {sorted(uniques)}")

if issues:
    print("\nISSUES FOUND:")
    for name, vals in issues:
        print(f"  {name}: unexpected values {vals}")
else:
    print("\nAll spot-checks passed!")

Dataset JSON loaded.
Labels       : {'background': 0, 'stone': 1}
Channel names: {'0': 'CT'}
Valid label values: [0, 1]
  neg_168.nii.gz: [np.int64(0)]
  stones_027.nii.gz: [np.int64(0), np.int64(1)]
  stones_040.nii.gz: [np.int64(0), np.int64(1)]
  neg_054.nii.gz: [np.int64(0)]
  neg_087.nii.gz: [np.int64(0)]

All spot-checks passed!


In [ ]:
import numpy as np
import nibabel as nib
from pathlib import Path
from collections import Counter
base = Path("/content/nnUNet_raw/Dataset502_KidneyStones")
labels_dir = base / "labelsTr"
images_dir = base / "imagesTr"
print("=" * 80)
print("🔍 VOXEL SPACING CHECK")
print("=" * 80)
spacing_issues = []
spacings = []
for img_path in sorted(images_dir.glob("*.nii.gz")):
    img = nib.load(str(img_path))
    pixdim = img.header.get_zooms()[:3]
    spacings.append(pixdim)
unique_spacings = Counter([tuple(np.round(s, 3)) for s in spacings])
print(f"Total images: {len(spacings)}")
print(f"Unique spacings found: {len(unique_spacings)}")
for spc, count in unique_spacings.most_common(10):
    print(f"  {spc} -> {count} cases")
if len(unique_spacings) > 1:
    print("\n⚠️  Multiple voxel spacings detected — nnUNet will resample during preprocessing.")
    spacing_issues.append("Multiple spacings (nnUNet handles this)")
print("\n" + "=" * 80)
print("🔍 STONE LABEL VERIFICATION (stone_*)")
print("=" * 80)
stone_files = sorted(labels_dir.glob("stone_*.nii.gz"))
bad_stones = []
for f in stone_files:
    if f.name.endswith(".bak"):
        continue

    data = np.asanyarray(nib.load(str(f)).dataobj).astype(np.uint8)

    kidney = int(np.sum(data == 1))
    stone = int(np.sum(data == 3))
    cyst = int(np.sum(data == 2))
    tumor = int(np.sum(data == 4))

    issues = []
    if stone > kidney and kidney > 0:
        issues.append("STONE>KIDNEY")
    if kidney == 0 and stone > 0:
        issues.append("NO_KIDNEY_BUT_HAS_STONE")
    if kidney < 100 and stone > 0:
        issues.append("TINY_KIDNEY")
    if stone > 500000:
        issues.append("HUGE_STONE")

    if issues:
        bad_stones.append((f.name, kidney, stone, cyst, tumor, issues))
        print(f"  ⚠️  {f.name:<20} K={kidney:>8,} S={stone:>8,} C={cyst:>6,} T={tumor:>6,} | {', '.join(issues)}")
    else:
        print(f"  ✅ {f.name:<20} K={kidney:>8,} S={stone:>8,} C={cyst:>6,} T={tumor:>6,}")
print("\n" + "=" * 80)
print("🔍 NEGATIVE CASE VERIFICATION (cyst_*, tumor_*, multi_*)")
print("=" * 80)
non_stone_files = sorted(labels_dir.glob("cyst_*.nii.gz")) + \
                  sorted(labels_dir.glob("tumor_*.nii.gz")) + \
                  sorted(labels_dir.glob("multi_*.nii.gz"))
neg_issues = []
for f in non_stone_files:
    data = np.asanyarray(nib.load(str(f)).dataobj).astype(np.uint8)
    stone = int(np.sum(data == 3))

    if stone > 0:
        neg_issues.append((f.name, stone))
        print(f"  ⚠️  {f.name:<25} unexpected stone voxels: {stone:,}")
print("\n" + "=" * 80)
print("📊 SUMMARY")
print("=" * 80)
print(f"Total images checked for spacing : {len(spacings)}")
print(f"Stone cases checked              : {len(stone_files)}")
print(f"Non-stone cases checked          : {len(non_stone_files)}")
if spacing_issues:
    for issue in spacing_issues:
        print(f"\n⚠️  {issue}")
else:
    print(f"\n✅ All images have consistent voxel spacing.")
if bad_stones:
    print(f"\n❌ STONE CASES WITH ISSUES: {len(bad_stones)}")
    for name, k, s, c, t, issues in bad_stones:
        print(f"   {name}: K={k:,} S={s:,} -> {', '.join(issues)}")
else:
    print(f"\n✅ All stone labels look anatomically correct.")
if neg_issues:
    print(f"\n❌ NON-STONE CASES WITH STONE VOXELS: {len(neg_issues)}")
    for name, stone in neg_issues:
        print(f"   {name}: {stone:,} stone voxels found")
else:
    print(f"\n✅ All non-stone cases are truly stone-negative.")
total_issues = len(bad_stones) + len(neg_issues)
print("\n" + "=" * 80)
if total_issues == 0:
    print("🎉 ALL CHECKS PASSED! Dataset is clean and ready for preprocessing.")
else:
    print(f"❌ FOUND {total_issues} ISSUE(S). Fix before preprocessing!")
print("=" * 80)

🔍 VOXEL SPACING CHECK
Total images: 290
Unique spacings found: 183
  (np.float32(0.742), np.float32(0.742), np.float32(1.25)) -> 33 cases
  (np.float32(0.703), np.float32(0.703), np.float32(1.25)) -> 9 cases
  (np.float32(0.5), np.float32(0.703), np.float32(0.703)) -> 6 cases
  (np.float32(0.838), np.float32(0.838), np.float32(1.25)) -> 4 cases
  (np.float32(3.0), np.float32(0.977), np.float32(0.977)) -> 4 cases
  (np.float32(2.0), np.float32(0.977), np.float32(0.977)) -> 4 cases
  (np.float32(1.0), np.float32(0.809), np.float32(0.809)) -> 4 cases
  (np.float32(1.0), np.float32(0.762), np.float32(0.762)) -> 4 cases
  (np.float32(0.807), np.float32(0.807), np.float32(1.25)) -> 3 cases
  (np.float32(0.787), np.float32(0.787), np.float32(1.25)) -> 3 cases

⚠️  Multiple voxel spacings detected — nnUNet will resample during preprocessing.

🔍 STONE LABEL VERIFICATION (stone_*)

🔍 NEGATIVE CASE VERIFICATION (cyst_*, tumor_*, multi_*)

📊 SUMMARY
Total images checked for spacing : 290
Stone cas

## Section 4: Planning & Preprocessing

nnUNet will extract the dataset fingerprint, compute normalization stats,
design the 3D U-Net architecture, and preprocess all training cases.

> **After this section completes, always run Cell 4.2 (tar-save) before switching runtimes.**

### Cell 4.0 — Disk Check

In [ ]:
!df -h /content

### Cell 4.1 — Plan & Preprocess

In [ ]:
import os, subprocess
from pathlib import Path

# ── FIX: nnUNet_raw must point to LOCAL SSD where the dataset was copied ──
os.environ["nnUNet_raw"]          = "/content/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"

# Results stay on Drive (so checkpoints survive runtime resets)
os.environ["nnUNet_results"] = "/content/drive/MyDrive/1THESIS_AMM/Dataset502_KidneyStones/nnUNet_results"

# Verify Dataset501 is actually there
dataset_path = Path("/content/nnUNet_raw/Dataset502_KidneyStones")
if dataset_path.exists():
    images = list((dataset_path / "imagesTr").glob("*.nii.gz"))
    labels = list((dataset_path / "labelsTr").glob("*.nii.gz"))
    print(f"✅ Dataset found: {len(images)} images, {len(labels)} labels")
else:
    print("❌ Dataset NOT found at", dataset_path)
    print("Re-run Cell 2.3 to copy from Drive first.")

print("\nActive env vars:")
for k in ["nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"]:
    print(f"  {k} = {os.environ[k]}")

❌ Dataset NOT found at /content/nnUNet_raw/Dataset502_KidneyStones
Re-run Cell 2.3 to copy from Drive first.

Active env vars:
  nnUNet_raw = /content/nnUNet_raw
  nnUNet_preprocessed = /content/nnUNet_preprocessed
  nnUNet_results = /content/drive/MyDrive/1THESIS_AMM/Dataset502_KidneyStones/nnUNet_results


In [ ]:
!nnUNetv2_preprocess -d 502 -c 3d_fullres -np 4

Preprocessing dataset Dataset502_KidneyStones
Configuration: 3d_fullres...
{'data_identifier': 'nnUNetPlans_3d_fullres', 'preprocessor_name': 'DefaultPreprocessor', 'batch_size': 2, 'patch_size': [128, 128, 128], 'median_image_size_in_voxels': [434.0, 512.0, 467.0], 'spacing': [0.9130857586860657, 0.78515625, 0.81640625], 'normalization_schemes': ['CTNormalization'], 'use_mask_for_norm': [False], 'resampling_fn_data': 'resample_data_or_seg_to_shape', 'resampling_fn_seg': 'resample_data_or_seg_to_shape', 'resampling_fn_data_kwargs': {'is_seg': False, 'order': 3, 'order_z': 0, 'force_separate_z': None}, 'resampling_fn_seg_kwargs': {'is_seg': True, 'order': 1, 'order_z': 0, 'force_separate_z': None}, 'resampling_fn_probabilities': 'resample_data_or_seg_to_shape', 'resampling_fn_probabilities_kwargs': {'is_seg': False, 'order': 1, 'order_z': 0, 'force_separate_z': None}, 'architecture': {'network_class_name': 'dynamic_network_architectures.architectures.unet.PlainConvUNet', 'arch_kwargs': 

### Cell 4.2 — 💾 Save Preprocessed Data as TAR to Drive

Run this **immediately after preprocessing finishes** and before any runtime switch.
The tar file is written directly to Drive so it survives session resets.

In [ ]:
import subprocess
from pathlib import Path

preprocessed_local = Path("/content/nnUNet_preprocessed")
tar_path = Path(TAR_PREPROCESSED_PATH)

# Verify source exists and has content
if not preprocessed_local.exists():
    raise FileNotFoundError("Local preprocessed folder not found — did preprocessing complete?")

contents = list(preprocessed_local.rglob("*"))
print(f"Files to archive: {len(contents)}")

# Run tar — write directly to Drive
print(f"Creating tar at: {tar_path}")
print("This may take a few minutes...")
result = subprocess.run(
    ["tar", "-cf", str(tar_path), "-C", "/content", "nnUNet_preprocessed"],
    capture_output=True, text=True
)
if result.returncode != 0:
    print("TAR FAILED:", result.stderr)
    raise RuntimeError("tar creation failed")

size_gb = tar_path.stat().st_size / 1e9
print(f"\nTAR saved successfully: {tar_path}")
print(f"Size: {size_gb:.2f} GB")
print("You can safely switch runtimes now.")

Files to archive: 1166
Creating tar at: /content/drive/MyDrive/1THESIS_AMM/Dataset502_KidneyStones/nnUNet_preprocessed_502.tar
This may take a few minutes...

TAR saved successfully: /content/drive/MyDrive/1THESIS_AMM/Dataset502_KidneyStones/nnUNet_preprocessed_502.tar
Size: 29.71 GB
You can safely switch runtimes now.


In [ ]:
from pathlib import Path

tar_path = Path("/content/drive/MyDrive/1THESIS_AMM/Dataset502_KidneyStones/nnUNet_preprocessed_502.tar")

print("Exists:", tar_path.exists())
if tar_path.exists():
    print("Size GB:", tar_path.stat().st_size / (1024**3))
    print("Path:", tar_path)

Exists: True
Size GB: 27.670869827270508
Path: /content/drive/MyDrive/1THESIS_AMM/Dataset502_KidneyStones/nnUNet_preprocessed_502.tar


### Cell 4.3 — 📦 Restore Preprocessed Data from TAR (after runtime reset)

Run this after a runtime reset **instead of re-running preprocessing**.
Make sure Sections 1–2 (install + mount + env vars) are run first.

In [ ]:
from pathlib import Path
import subprocess

tar_src = Path("/content/drive/MyDrive/1THESIS_AMM/Dataset502_KidneyStones/nnUNet_preprocessed_502.tar")

print(f"Size: {tar_src.stat().st_size / 1e9:.2f} GB")

# Check actual file type (ignore the extension)
result = subprocess.run(["file", str(tar_src)], capture_output=True, text=True)
print(result.stdout)

# Peek at first few bytes
with open(tar_src, "rb") as f:
    print("Magic bytes:", f.read(8).hex())

Size: 29.71 GB
/content/drive/MyDrive/1THESIS_AMM/Dataset502_KidneyStones/nnUNet_preprocessed_502.tar: POSIX tar archive (GNU)

Magic bytes: 6e6e554e65745f70


In [ ]:
import tarfile, os
from pathlib import Path

tar_src = Path("/content/drive/MyDrive/1THESIS_AMM/Dataset502_KidneyStones/nnUNet_preprocessed_502.tar")
dst = Path("/content/nnUNet_preprocessed")

if dst.exists():
    print("Already exists locally — skipping extraction.")
else:
    print("Extracting preprocessed tar to local SSD...")
    with tarfile.open(tar_src, "r") as tf:
        tf.extractall("/content")
    print("Done.")

os.environ["nnUNet_preprocessed"] = str(dst)

# Verify
dataset = dst / "Dataset502_KidneyStones"
files = list(dataset.rglob("*"))
plans = dataset / "nnUNetPlans.json"
print(f"Files: {len(files)}")
print(f"nnUNetPlans.json: {plans.exists()}")
print(f"nnUNet_preprocessed → {os.environ['nnUNet_preprocessed']}")

Extracting preprocessed tar to local SSD...


/tmp/ipykernel_977/1372635572.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall("/content")


Done.
Files: 1165
nnUNetPlans.json: True
nnUNet_preprocessed → /content/nnUNet_preprocessed


In [ ]:
!pip install nnunetv2 -q

### Cell 4.4 — (Optional) Patch Batch Size in Plans

In [ ]:
import json, shutil
from pathlib import Path

plans_path = Path("/content/nnUNet_preprocessed/Dataset502_KidneyStones/nnUNetPlans.json")
backup_path = plans_path.with_suffix(".json.backup")

if not backup_path.exists():
    shutil.copy2(plans_path, backup_path)
    print("Backup created:", backup_path)

with open(plans_path) as f:
    plans = json.load(f)

old = plans["configurations"]["3d_fullres"]["batch_size"]
print("Current batch_size:", old)

# Uncomment to change:
plans["configurations"]["3d_fullres"]["batch_size"] = 2
with open(plans_path, "w") as f:
     json.dump(plans, f, indent=4)
print("Updated to:", 2)

# Display 3d_fullres config
print("\n=== 3d_fullres config ===")
print(json.dumps(plans["configurations"]["3d_fullres"], indent=2))

Backup created: /content/nnUNet_preprocessed/Dataset502_KidneyStones/nnUNetPlans.json.backup
Current batch_size: 2
Updated to: 2

=== 3d_fullres config ===
{
  "data_identifier": "nnUNetPlans_3d_fullres",
  "preprocessor_name": "DefaultPreprocessor",
  "batch_size": 2,
  "patch_size": [
    128,
    128,
    128
  ],
  "median_image_size_in_voxels": [
    434.0,
    512.0,
    467.0
  ],
  "spacing": [
    0.9130857586860657,
    0.78515625,
    0.81640625
  ],
  "normalization_schemes": [
    "CTNormalization"
  ],
  "use_mask_for_norm": [
    false
  ],
  "resampling_fn_data": "resample_data_or_seg_to_shape",
  "resampling_fn_seg": "resample_data_or_seg_to_shape",
  "resampling_fn_data_kwargs": {
    "is_seg": false,
    "order": 3,
    "order_z": 0,
    "force_separate_z": null
  },
  "resampling_fn_seg_kwargs": {
    "is_seg": true,
    "order": 1,
    "order_z": 0,
    "force_separate_z": null
  },
  "resampling_fn_probabilities": "resample_data_or_seg_to_shape",
  "resampling_fn_

In [ ]:
from pathlib import Path
from collections import Counter

folder = Path("/content/nnUNet_preprocessed/Dataset502_KidneyStones/nnUNetPlans_3d_fullres")

print(Counter(p.suffix for p in folder.iterdir() if p.is_file()))

print("\nSample files:")
for p in sorted(folder.iterdir())[:30]:
    print(p.name)

Counter({'.b2nd': 580, '.pkl': 290})

Sample files:
neg_000.b2nd
neg_000.pkl
neg_000_seg.b2nd
neg_001.b2nd
neg_001.pkl
neg_001_seg.b2nd
neg_002.b2nd
neg_002.pkl
neg_002_seg.b2nd
neg_003.b2nd
neg_003.pkl
neg_003_seg.b2nd
neg_004.b2nd
neg_004.pkl
neg_004_seg.b2nd
neg_005.b2nd
neg_005.pkl
neg_005_seg.b2nd
neg_006.b2nd
neg_006.pkl
neg_006_seg.b2nd
neg_007.b2nd
neg_007.pkl
neg_007_seg.b2nd
neg_008.b2nd
neg_008.pkl
neg_008_seg.b2nd
neg_009.b2nd
neg_009.pkl
neg_009_seg.b2nd


In [ ]:
from pathlib import Path

folder = Path("/content/nnUNet_preprocessed/Dataset502_KidneyStones/nnUNetPlans_3d_fullres")

for f in folder.glob("*.ini"):
    print("Deleting:", f)
    f.unlink()

print("Done. Remaining .ini files:", list(folder.glob("*.ini")))

Done. Remaining .ini files: []


In [ ]:
# ============================================================
# nnU-Net v2 B2ND PREPROCESSED CHECKER
# For folders containing:
#   case_id.b2nd
#   case_id_seg.b2nd
#   case_id.pkl
#
# Checks:
# 1. Case count
# 2. Matching image/seg/pkl files
# 3. Voxel counts per label from *_seg.b2nd
# 4. Obvious label swap / wrong-class flags
# ============================================================

from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import gc

# Install if needed
try:
    import blosc2
except ImportError:
    !pip install blosc2 -q
    import blosc2

# ----------------------------
# Settings
# ----------------------------
EXPECTED_CASES = 290

ROOT = Path("/content/nnUNet_preprocessed")

LABELS = {
    0: "background",
    1: "kidney",
    2: "cyst",
    3: "stone",
    4: "tumor",
}

REPORT_DIR = Path("/content/drive/MyDrive/1THESIS_AMM/Dataset502_KidneyStones/label_check_reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Find 3d_fullres folder
# ----------------------------
folders = sorted(ROOT.rglob("nnUNetPlans_3d_fullres"))

print("=" * 90)
print("FINDING 3D FULLRES PREPROCESSED FOLDERS")
print("=" * 90)

if not folders:
    raise FileNotFoundError("No nnUNetPlans_3d_fullres folder found under /content/nnUNet_preprocessed")

for folder in folders:
    b2nd_all = sorted(folder.glob("*.b2nd"))
    seg_b2nd = sorted(folder.glob("*_seg.b2nd"))
    img_b2nd = [p for p in b2nd_all if not p.name.endswith("_seg.b2nd")]
    pkl_files = sorted(folder.glob("*.pkl"))

    print(folder)
    print(f"  image .b2nd     : {len(img_b2nd)}")
    print(f"  seg _seg.b2nd   : {len(seg_b2nd)}")
    print(f"  .pkl            : {len(pkl_files)}")
    print()

# Pick folder with most segmentation files
PREPROCESSED_3D = max(folders, key=lambda f: len(list(f.glob("*_seg.b2nd"))))

print("Using folder:")
print(PREPROCESSED_3D)

# ----------------------------
# Case count check
# ----------------------------
all_b2nd = sorted(PREPROCESSED_3D.glob("*.b2nd"))
seg_files = sorted(PREPROCESSED_3D.glob("*_seg.b2nd"))
img_files = [p for p in all_b2nd if not p.name.endswith("_seg.b2nd")]
pkl_files = sorted(PREPROCESSED_3D.glob("*.pkl"))

seg_cases = {p.name.replace("_seg.b2nd", "") for p in seg_files}
img_cases = {p.name.replace(".b2nd", "") for p in img_files}
pkl_cases = {p.name.replace(".pkl", "") for p in pkl_files}

missing_img = sorted(seg_cases - img_cases)
missing_seg = sorted(img_cases - seg_cases)
missing_pkl_from_seg = sorted(seg_cases - pkl_cases)
missing_seg_from_pkl = sorted(pkl_cases - seg_cases)

print("\n" + "=" * 90)
print("CASE COUNT CHECK")
print("=" * 90)

print(f"Expected cases       : {EXPECTED_CASES}")
print(f"Image .b2nd files    : {len(img_files)}")
print(f"Seg _seg.b2nd files  : {len(seg_files)}")
print(f".pkl files           : {len(pkl_files)}")

if len(seg_files) == EXPECTED_CASES and len(img_files) == EXPECTED_CASES and len(pkl_files) == EXPECTED_CASES:
    print("\n✅ Case count looks complete.")
else:
    print("\n⚠️ Case count mismatch. Review missing files below.")

if missing_img:
    print(f"\n❌ Seg exists but image .b2nd missing: {len(missing_img)}")
    for case in missing_img[:30]:
        print("  missing image:", case)

if missing_seg:
    print(f"\n❌ Image exists but _seg.b2nd missing: {len(missing_seg)}")
    for case in missing_seg[:30]:
        print("  missing seg:", case)

if missing_pkl_from_seg:
    print(f"\n❌ Seg exists but .pkl missing: {len(missing_pkl_from_seg)}")
    for case in missing_pkl_from_seg[:30]:
        print("  missing pkl:", case)

if missing_seg_from_pkl:
    print(f"\n❌ .pkl exists but _seg.b2nd missing: {len(missing_seg_from_pkl)}")
    for case in missing_seg_from_pkl[:30]:
        print("  missing seg:", case)

if len(seg_files) == 0:
    raise FileNotFoundError("No *_seg.b2nd files found. Cannot check label voxel counts.")

# ----------------------------
# Voxel label count check
# ----------------------------
print("\n" + "=" * 90)
print("VOXEL LABEL CHECK FROM *_seg.b2nd FILES")
print("=" * 90)

rows = []
flagged = []

for i, f in enumerate(seg_files):
    case_id = f.name.replace("_seg.b2nd", "")

    try:
        seg_b2nd = blosc2.open(str(f), mode="r")
        seg = seg_b2nd[:]  # loads only the segmentation file, not the image data

        # Seg may be shaped like (1, x, y, z) or (x, y, z)
        if seg.ndim == 4 and seg.shape[0] == 1:
            seg = seg[0]

        uniques, counts = np.unique(seg, return_counts=True)
        uniques_int = [int(round(float(u))) for u in uniques]
        count_map = dict(zip(uniques_int, counts.astype(int)))

        row = {
            "case": case_id,
            "seg_file": f.name,
            "seg_shape": tuple(seg.shape),
            "background_0": count_map.get(0, 0),
            "kidney_1": count_map.get(1, 0),
            "cyst_2": count_map.get(2, 0),
            "stone_3": count_map.get(3, 0),
            "tumor_4": count_map.get(4, 0),
            "unique_labels": sorted(uniques_int),
        }

        issues = []

        # Unexpected labels
        unexpected = set(uniques_int) - set(LABELS.keys())
        if unexpected:
            issues.append(f"Unexpected labels: {sorted(unexpected)}")

        name = case_id.lower()

        # Prefix-based checks
        if name.startswith("stone_"):
            if row["stone_3"] == 0:
                issues.append("stone_* case has ZERO stone label 3")
            if row["kidney_1"] == 0:
                issues.append("stone_* case has ZERO kidney label 1")
            if row["cyst_2"] > 0:
                issues.append("stone_* case has cyst label 2")
            if row["tumor_4"] > 0:
                issues.append("stone_* case has tumor label 4")

        elif name.startswith("cyst_"):
            if row["cyst_2"] == 0:
                issues.append("cyst_* case has ZERO cyst label 2")
            if row["kidney_1"] == 0:
                issues.append("cyst_* case has ZERO kidney label 1")
            if row["stone_3"] > 0:
                issues.append("cyst_* case has stone label 3")
            if row["tumor_4"] > 0:
                issues.append("cyst_* case has tumor label 4")

        elif name.startswith("tumor_"):
            if row["tumor_4"] == 0:
                issues.append("tumor_* case has ZERO tumor label 4")
            if row["kidney_1"] == 0:
                issues.append("tumor_* case has ZERO kidney label 1")
            if row["stone_3"] > 0:
                issues.append("tumor_* case has stone label 3")
            if row["cyst_2"] > 0:
                issues.append("tumor_* case has cyst label 2")

        elif name.startswith("multi_"):
            # Multiclass cases can contain multiple abnormalities.
            if row["kidney_1"] == 0 and (
                row["cyst_2"] > 0 or row["stone_3"] > 0 or row["tumor_4"] > 0
            ):
                issues.append("multi_* case has abnormality but ZERO kidney label 1")

        else:
            issues.append("Unknown prefix; cannot infer expected class from filename")

        # General sanity checks
        if row["kidney_1"] == 0 and (
            row["cyst_2"] > 0 or row["stone_3"] > 0 or row["tumor_4"] > 0
        ):
            issues.append("Abnormality exists but kidney label 1 is missing")

        if row["stone_3"] > 0:
            if row["kidney_1"] > 0 and row["stone_3"] > row["kidney_1"]:
                issues.append("Stone voxels greater than kidney voxels")
            if row["stone_3"] > 500000:
                issues.append("Very large stone voxel count")

        row["issues"] = " | ".join(issues)

        rows.append(row)

        if issues:
            flagged.append(row)

        if (i + 1) % 25 == 0 or (i + 1) == len(seg_files):
            print(f"Checked {i + 1}/{len(seg_files)} segmentation files...")

        del seg, seg_b2nd
        gc.collect()

    except Exception as e:
        flagged.append({
            "case": case_id,
            "seg_file": f.name,
            "issues": f"Could not read/check _seg.b2nd: {e}",
        })

# ----------------------------
# Save report
# ----------------------------
df = pd.DataFrame(rows)

report_path = REPORT_DIR / "preprocessed_b2nd_seg_voxel_counts.csv"
df.to_csv(report_path, index=False)

print("\n" + "=" * 90)
print("SUMMARY")
print("=" * 90)

print(f"Segmentation cases checked: {len(df)}")
print(f"Flagged cases             : {len(flagged)}")
print(f"Report saved              : {report_path}")

print("\nClass presence summary:")
for col in ["kidney_1", "cyst_2", "stone_3", "tumor_4"]:
    if col in df.columns:
        present_cases = int((df[col] > 0).sum())
        total_voxels = int(df[col].sum())
        print(f"  {col:<10} | cases present: {present_cases:<4} | total voxels: {total_voxels:,}")

print("\nUnique label combinations:")
if len(df) > 0:
    combo_counter = Counter(tuple(x) for x in df["unique_labels"])
    for combo, count in combo_counter.most_common(20):
        print(f"  {combo} -> {count} cases")

if flagged:
    print("\n⚠️ FLAGGED CASES:")
    flagged_df = df[df["issues"] != ""].copy()
    display(flagged_df)
else:
    print("\n🎉 ALL GOOD: Case count and label voxel checks look clean.")

print("=" * 90)

FINDING 3D FULLRES PREPROCESSED FOLDERS
/content/nnUNet_preprocessed/Dataset502_KidneyStones/nnUNetPlans_3d_fullres
  image .b2nd     : 290
  seg _seg.b2nd   : 290
  .pkl            : 290

Using folder:
/content/nnUNet_preprocessed/Dataset502_KidneyStones/nnUNetPlans_3d_fullres

CASE COUNT CHECK
Expected cases       : 290
Image .b2nd files    : 290
Seg _seg.b2nd files  : 290
.pkl files           : 290

✅ Case count looks complete.

VOXEL LABEL CHECK FROM *_seg.b2nd FILES
Checked 25/290 segmentation files...
Checked 50/290 segmentation files...
Checked 75/290 segmentation files...
Checked 100/290 segmentation files...
Checked 125/290 segmentation files...
Checked 150/290 segmentation files...
Checked 175/290 segmentation files...
Checked 200/290 segmentation files...
Checked 225/290 segmentation files...
Checked 250/290 segmentation files...
Checked 275/290 segmentation files...
Checked 290/290 segmentation files...

SUMMARY
Segmentation cases checked: 290
Flagged cases             : 29

,case,seg_file,seg_shape,background_0,kidney_1,cyst_2,stone_3,tumor_4,unique_labels,issues
0,neg_000,neg_000_seg.b2nd,"(522, 576, 554)",166572080,0,0,0,0,"[-1, 0]",Unexpected labels: [-1] | Unknown prefix; cann...
1,neg_001,neg_001_seg.b2nd,"(259, 526, 506)",68934023,0,0,0,0,"[-1, 0]",Unexpected labels: [-1] | Unknown prefix; cann...
2,neg_002,neg_002_seg.b2nd,"(215, 484, 465)",48387681,0,0,0,0,"[-1, 0]",Unexpected labels: [-1] | Unknown prefix; cann...
3,neg_003,neg_003_seg.b2nd,"(286, 507, 488)",70760701,0,0,0,0,"[-1, 0]",Unexpected labels: [-1] | Unknown prefix; cann...
4,neg_004,neg_004_seg.b2nd,"(242, 484, 465)",54464338,0,0,0,0,"[-1, 0]",Unexpected labels: [-1] | Unknown prefix; cann...
...,...,...,...,...,...,...,...,...,...,...
285,stones_070,stones_070_seg.b2nd,"(187, 401, 386)",28944716,118,0,0,0,"[-1, 0, 1]",Unexpected labels: [-1] | Unknown prefix; cann...
286,stones_071,stones_071_seg.b2nd,"(555, 488, 469)",127023099,461,0,0,0,"[-1, 0, 1]",Unexpected labels: [-1] | Unknown prefix; cann...
287,stones_072,stones_072_seg.b2nd,"(215, 448, 431)",41513572,180,0,0,0,"[-1, 0, 1]",Unexpected labels: [-1] | Unknown prefix; cann...
288,stones_073,stones_073_seg.b2nd,"(434, 484, 465)",97675588,167,0,0,0,"[-1, 0, 1]",Unexpected labels: [-1] | Unknown prefix; cann...


## Section 5: Model Training (3D Full-Res)

Train all 5 folds. After each fold, checkpoints are on Drive under `nnUNet_results/`.

| GPU | Time per fold (approx) |
|-----|------------------------|
| T4  | 4–6 h |
| L4  | 2–3 h |
| A100 | 1–2 h |

> **Resume behaviour:** if a fold was interrupted, add `--c` to resume from last checkpoint.

### Cell 5.1 — Train Fold 0

In [ ]:
!nnUNetv2_train 502 3d_fullres 0 --npz --c

Streaming output truncated to the last 5000 lines.
2026-05-12 04:04:45.522301: Current learning rate: 0.00723
2026-05-12 04:05:12.215124: train_loss -0.7219
2026-05-12 04:05:12.219895: val_loss -0.7491
2026-05-12 04:05:12.224014: Pseudo dice [np.float32(0.0)]
2026-05-12 04:05:12.227589: Epoch time: 26.7 s
2026-05-12 04:05:13.627063: 
2026-05-12 04:05:13.630274: Epoch 304
2026-05-12 04:05:13.633295: Current learning rate: 0.00722
2026-05-12 04:05:40.145136: train_loss -0.7679
2026-05-12 04:05:40.148863: val_loss -0.7529
2026-05-12 04:05:40.152152: Pseudo dice [np.float32(0.0)]
2026-05-12 04:05:40.156115: Epoch time: 26.52 s
2026-05-12 04:05:41.588261: 
2026-05-12 04:05:41.591483: Epoch 305
2026-05-12 04:05:41.594575: Current learning rate: 0.00721
2026-05-12 04:06:08.608819: train_loss -0.7281
2026-05-12 04:06:08.613219: val_loss -0.7858
2026-05-12 04:06:08.617565: Pseudo dice [np.float32(0.0)]
2026-05-12 04:06:08.623813: Epoch time: 27.02 s
2026-05-12 04:06:10.043345: 
2026-05-12 04:06

### Cell 5.2 — Train Fold 1

In [ ]:
!nnUNetv2_train 502 3d_fullres 1 --npz --c


############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-05-12 12:30:27.055169: Using torch.compile...
2026-05-12 12:30:28.599068: do_dummy_2d_data_aug: False
2026-05-12 12:30:28.613597: Using splits from existing split file: /content/nnUNet_preprocessed/Dataset502_KidneyStones/splits_final.json
2026-05-12 12:30:28.616469: The split file contains

### Cell 5.3 — Train Fold 2

In [ ]:
!nnUNetv2_train 502 3d_fullres 2 --npz

Streaming output truncated to the last 5000 lines.
2026-05-12 16:28:19.520461: Current learning rate: 0.00723
2026-05-12 16:28:46.918758: train_loss -0.723
2026-05-12 16:28:46.923481: val_loss -0.8376
2026-05-12 16:28:46.929488: Pseudo dice [np.float32(0.0)]
2026-05-12 16:28:46.933130: Epoch time: 27.41 s
2026-05-12 16:28:48.341076: 
2026-05-12 16:28:48.343916: Epoch 304
2026-05-12 16:28:48.346882: Current learning rate: 0.00722
2026-05-12 16:29:15.450567: train_loss -0.694
2026-05-12 16:29:15.454563: val_loss -0.8847
2026-05-12 16:29:15.458422: Pseudo dice [np.float32(0.0)]
2026-05-12 16:29:15.461904: Epoch time: 27.11 s
2026-05-12 16:29:16.915462: 
2026-05-12 16:29:16.918600: Epoch 305
2026-05-12 16:29:16.921600: Current learning rate: 0.00721
2026-05-12 16:29:43.080019: train_loss -0.7182
2026-05-12 16:29:43.100918: val_loss -0.8621
2026-05-12 16:29:43.105903: Pseudo dice [np.float32(0.0)]
2026-05-12 16:29:43.110081: Epoch time: 26.17 s
2026-05-12 16:29:44.523321: 
2026-05-12 16:29:

### Cell 5.4 — Train Fold 3

In [ ]:
#!nnUNetv2_train 502 3d_fullres 3 --npz --c


############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-05-11 01:29:02.767830: Using torch.compile...
2026-05-11 01:29:05.551616: do_dummy_2d_data_aug: False
2026-05-11 01:29:05.556978: Using splits from existing split file: /content/nnUNet_preprocessed/Dataset503_KidneyTumorFixed/splits_final.json
2026-05-11 01:29:05.560371: The split file cont

### Cell 5.5 — Train Fold 4

In [ ]:
!nnUNetv2_train 502 3d_fullres 4 --npz

Streaming output truncated to the last 5000 lines.
2026-05-13 00:51:38.686142: Current learning rate: 0.00723
2026-05-13 00:52:05.012539: train_loss -0.7115
2026-05-13 00:52:05.017612: val_loss -0.6792
2026-05-13 00:52:05.021185: Pseudo dice [np.float32(0.0)]
2026-05-13 00:52:05.024900: Epoch time: 26.34 s
2026-05-13 00:52:06.431194: 
2026-05-13 00:52:06.434822: Epoch 304
2026-05-13 00:52:06.438753: Current learning rate: 0.00722
2026-05-13 00:52:34.340741: train_loss -0.7393
2026-05-13 00:52:34.345667: val_loss -0.5636
2026-05-13 00:52:34.350082: Pseudo dice [np.float32(0.0)]
2026-05-13 00:52:34.355150: Epoch time: 27.91 s
2026-05-13 00:52:35.776886: 
2026-05-13 00:52:35.780167: Epoch 305
2026-05-13 00:52:35.783841: Current learning rate: 0.00721
2026-05-13 00:53:03.223085: train_loss -0.7312
2026-05-13 00:53:03.229409: val_loss -0.772
2026-05-13 00:53:03.238002: Pseudo dice [np.float32(0.0)]
2026-05-13 00:53:03.242047: Epoch time: 27.45 s
2026-05-13 00:53:04.629272: 
2026-05-13 00:53

### Cell 5.6 — Manual Resume (specific fold)

Use this if a fold was interrupted and you need to resume it specifically.

In [ ]:
# Uncomment the fold you need to resume:

# !nnUNetv2_train 501 3d_fullres 0 --c --npz --num_epochs 200
# !nnUNetv2_train 501 3d_fullres 1 --c --npz --num_epochs 200
# !nnUNetv2_train 501 3d_fullres 2 --c --npz --num_epochs 200
# !nnUNetv2_train 501 3d_fullres 3 --c --npz --num_epochs 200
# !nnUNetv2_train 501 3d_fullres 4 --c --npz --num_epochs 200

# Validation only (run after fold completes):
# !nnUNetv2_train 501 3d_fullres 0 --val --npz

## Section 6: Find Best Configuration

After all 5 folds complete, find the best configuration and postprocessing.

In [ ]:
import os
os.environ["nnUNet_results"] = "/content/drive/MyDrive/1THESIS_AMM/nnUNet_results"
!ls -lah /content/drive/MyDrive/1THESIS_AMM/nnUNet_results/Dataset502_KidneyStones

In [ ]:
!nnUNetv2_find_best_configuration 501 \
  -c 3d_fullres \
  -tr nnUNetTrainer \
  -p nnUNetPlans \
  --disable_ensembling

In [ ]:
!cat /content/drive/MyDrive/1THESIS_AMM/nnUNet_results/Dataset501_KidneyStones/inference_instructions.txt

## Section 7: 5-Fold Cross-Validation Evaluation

Generate per-fold validation predictions and compute per-class metrics.
Produces `cv_summary.json` for your thesis.

### Cell 7.1 — Generate Validation Predictions for All Folds

In [ ]:
import os, json, subprocess, shutil
from pathlib import Path

dataset_id = 501
config = "3d_fullres"
preprocessed_dir = Path(f"/content/nnUNet_preprocessed/Dataset{dataset_id}_KidneyStones")
raw_dir = Path(f"/content/nnUNet_raw/Dataset{dataset_id}_KidneyStones")
results_dir = Path(f"/content/drive/MyDrive/1THESIS_AMM/nnUNet_results/Dataset{dataset_id}_KidneyStones")

# Auto-detect trainer folder
trainer_candidates = list(results_dir.glob("nnUNetTrainer__*__3d_fullres"))
assert len(trainer_candidates) > 0, "No trained model found! Complete Section 5 first."
trainer_dir = trainer_candidates[0]
print("Trainer folder:", trainer_dir.name)

# Load splits
splits_file = preprocessed_dir / "splits_final.json"
assert splits_file.exists(), f"{splits_file} not found!"
with open(splits_file) as f:
    splits = json.load(f)

cv_base = Path("/content/cv_predictions")
cv_base.mkdir(exist_ok=True)

for fold in range(5):
    print(f"\n=== Fold {fold} ===")
    val_cases = splits[fold]['val']
    print(f"Validation cases: {len(val_cases)}")

    fold_in  = cv_base / f"fold_{fold}_input"
    fold_out = cv_base / f"fold_{fold}_output"
    fold_in.mkdir(exist_ok=True)
    fold_out.mkdir(exist_ok=True)

    # Clean old symlinks
    for p in list(fold_in.iterdir()):
        p.unlink()

    missing = 0
    for case in val_cases:
        src = raw_dir / "imagesTr" / f"{case}_0000.nii.gz"
        dst = fold_in / f"{case}_0000.nii.gz"
        if src.exists():
            if dst.exists(): dst.unlink()
            dst.symlink_to(src)
        else:
            missing += 1
            print(f"  MISSING: {src.name}")
    if missing:
        print(f"  WARNING: {missing} cases missing.")

    # Run prediction for this fold
    cmd = [
        "nnUNetv2_predict",
        "-i", str(fold_in),
        "-o", str(fold_out),
        "-d", str(dataset_id),
        "-c", config,
        "-f", str(fold),
        "-chk", "checkpoint_best.pth",
        "-npp", "1",
        "-nps", "1",
        "--verbose"
    ]
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("ERROR!")
        print(result.stdout[-1000:])
        print(result.stderr[-1000:])
    else:
        print("Prediction complete.")

    # Merge into single folder
    merged = cv_base / "merged"
    merged.mkdir(exist_ok=True)
    for pred in fold_out.glob("*.nii.gz"):
        shutil.copy2(pred, merged / pred.name)

print(f"\nAll predictions merged into: {cv_base / 'merged'}")
print(f"Total files: {len(list((cv_base / 'merged').glob('*.nii.gz')))}")

### Cell 7.2 — Compute Per-Class Metrics & Save summary.json

In [ ]:
!pip install medpy scikit-learn -q

In [ ]:
import json, numpy as np, nibabel as nib
from pathlib import Path
from tqdm.notebook import tqdm
from medpy.metric.binary import hd95
from sklearn.metrics import precision_score, recall_score, f1_score

pred_dir   = Path("/content/cv_predictions/merged")
label_dir  = Path("/content/nnUNet_raw/Dataset501_KidneyStones/labelsTr")
output_json = Path("/content/drive/MyDrive/1THESIS_AMM/nnUNet_results/Dataset501_KidneyStones/cv_summary.json")
output_json.parent.mkdir(parents=True, exist_ok=True)

# ── Read class names from dataset.json ────────────────────────────────
with open("/content/nnUNet_raw/Dataset501_KidneyStones/dataset.json") as f:
    meta = json.load(f)
raw_labels = meta.get("labels", {})
if isinstance(raw_labels, dict):
    # nnUNet v2 format: {"background": 0, "stone": 1, ...}
    class_names = {int(v): k for k, v in raw_labels.items() if int(v) != 0}
else:
    class_names = {i: f"class_{i}" for i in range(1, len(raw_labels))}
classes = sorted(class_names.keys())
print("Evaluating classes:", class_names)

results = {c: {"DSC": [], "IoU": [], "HD95": [], "Precision": [], "Recall": [], "F1": []} for c in classes}
per_case_results = []

pred_files = sorted(pred_dir.glob("*.nii.gz"))
print(f"Found {len(pred_files)} predictions.")

for pf in tqdm(pred_files):
    case_id = pf.name.replace(".nii.gz", "")
    lf = label_dir / f"{case_id}.nii.gz"
    if not lf.exists():
        print(f"Skipping {case_id}: label not found.")
        continue

    pred  = np.asanyarray(nib.load(pf).dataobj).astype(np.uint8)
    label = np.asanyarray(nib.load(lf).dataobj).astype(np.uint8)
    case_entry = {"case_id": case_id}

    for c in classes:
        p = (pred  == c).astype(np.uint8)
        l = (label == c).astype(np.uint8)

        if l.sum() == 0 and p.sum() == 0:
            dsc, iou, precision, recall, f1 = 1.0, 1.0, 1.0, 1.0, 1.0
            hd = None
        elif l.sum() == 0 or p.sum() == 0:
            dsc, iou, precision, recall, f1 = 0.0, 0.0, 0.0, 0.0, 0.0
            hd = None
        else:
            inter = np.logical_and(p, l).sum()
            union = np.logical_or(p, l).sum()
            dsc   = float(2.0 * inter / (p.sum() + l.sum()))
            iou   = float(inter / union)
            p_flat, l_flat = p.flatten(), l.flatten()
            precision = float(precision_score(l_flat, p_flat, zero_division=0))
            recall    = float(recall_score(l_flat, p_flat, zero_division=0))
            f1        = float(f1_score(l_flat, p_flat, zero_division=0))
            try:    hd = float(hd95(p, l))
            except: hd = None

        for metric, val in [("DSC", dsc), ("IoU", iou), ("HD95", hd),
                             ("Precision", precision), ("Recall", recall), ("F1", f1)]:
            results[c][metric].append(val)
        case_entry[class_names[c]] = {
            "DSC": dsc, "IoU": iou, "HD95": hd,
            "Precision": precision, "Recall": recall, "F1": f1
        }
    per_case_results.append(case_entry)

# Aggregate
summary = {"per_case": per_case_results, "aggregate": {}}
for c in classes:
    agg = {}
    for metric, vals in results[c].items():
        clean = [v for v in vals if v is not None]
        agg[f"{metric}_mean"] = float(np.mean(clean)) if clean else None
        agg[f"{metric}_std"]  = float(np.std(clean))  if clean else None
    summary["aggregate"][class_names[c]] = agg

with open(output_json, "w") as f:
    json.dump(summary, f, indent=2)

print("Summary saved to:", output_json)
print("\nAggregate Metrics:")
for cls_name, metrics in summary["aggregate"].items():
    print(f"  {cls_name}:")
    for k, v in metrics.items():
        print(f"    {k}: {v:.4f}" if v is not None else f"    {k}: N/A")

## Section 8: Custom Inference with Profiling

5-fold ensemble inference on any volume(s) with per-volume latency and peak VRAM logging.

In [ ]:
import torch, json, numpy as np
from pathlib import Path
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
from nnunetv2.imageio.simpleitk_reader_writer import SimpleITKIO

# ── USER CONFIG ───────────────────────────────────────────────────────
input_path     = Path("/content/nnUNet_raw/Dataset501_KidneyStones/imagesTr")
output_folder  = Path("/content/inference_output_501")
checkpoint_name = "checkpoint_best.pth"
# ─────────────────────────────────────────────────────────────────────

output_folder.mkdir(exist_ok=True)

results_dir = Path("/content/drive/MyDrive/1THESIS_AMM/nnUNet_results/Dataset501_KidneyStones")
candidates  = list(results_dir.glob("nnUNetTrainer__*__3d_fullres"))
if not candidates:
    raise RuntimeError("No trained model found!")
model_folder = candidates[0]
print("Model folder:", model_folder.name)

predictor = nnUNetPredictor(
    tile_step_size=0.5, use_gaussian=True, use_mirroring=True,
    device=torch.device('cuda', 0), verbose=False, allow_tqdm=True
)
predictor.initialize_from_trained_model_folder(
    str(model_folder), use_folds=(0, 1, 2, 3, 4), checkpoint_name=checkpoint_name
)
print("Predictor ready.")

files = [input_path] if input_path.is_file() else sorted(input_path.glob("*.nii.gz"))
if not files: raise ValueError(f"No .nii.gz files in {input_path}")

io = SimpleITKIO()
profile = []

for f in files:
    print(f"Processing: {f.name}")
    image, props = io.read_images([str(f)])
    torch.cuda.reset_peak_memory_stats()
    s = torch.cuda.Event(enable_timing=True)
    e = torch.cuda.Event(enable_timing=True)
    torch.cuda.synchronize(); s.record()
    ret = predictor.predict_single_npy_array(image, props, None, None, False)
    seg = ret[0] if isinstance(ret, tuple) else ret
    e.record(); torch.cuda.synchronize()
    latency = s.elapsed_time(e) / 1000
    vram    = torch.cuda.max_memory_allocated() / 1e9
    out_name = f.name.replace("_0000.nii.gz", ".nii.gz")
    io.write_seg(seg.astype(np.uint8), str(output_folder / out_name), props)
    entry = {"file": f.name, "latency_seconds": round(latency, 4), "peak_vram_gb": round(vram, 4)}
    profile.append(entry)
    print(f"  Latency: {latency:.3f}s  |  Peak VRAM: {vram:.2f} GB")

with open(output_folder / "inference_profile.json", "w") as f:
    json.dump(profile, f, indent=2)
print(f"\nDone. Profile saved to {output_folder}/inference_profile.json")

## Section 9: Save Results to Google Drive

Back up all training outputs (checkpoints, plans, CV predictions) to Drive.

In [ ]:
from pathlib import Path
import shutil

src = Path("/content/nnUNet_results")
dst = Path("/content/drive/MyDrive/1THESIS_AMM/nnUNet_results")
dst.mkdir(parents=True, exist_ok=True)

print("Syncing nnUNet_results to Drive...")
for item in src.iterdir():
    dest_item = dst / item.name
    if item.is_dir():
        if dest_item.exists(): shutil.rmtree(dest_item)
        shutil.copytree(item, dest_item)
    else:
        shutil.copy2(item, dest_item)
    print(f"  Copied: {item.name}")
print("Done.")

## Section 10: Export Model for Deployment

Create a clean export folder with only the files needed for Streamlit/inference.

In [ ]:
from pathlib import Path
import shutil, json

dataset_raw  = Path("/content/nnUNet_raw/Dataset501_KidneyStones")
results_dir  = Path("/content/drive/MyDrive/1THESIS_AMM/nnUNet_results/Dataset501_KidneyStones")
export_dir   = Path("/content/streamlit_export_501")
export_dir.mkdir(parents=True, exist_ok=True)

shutil.copy2(dataset_raw / "dataset.json", export_dir / "dataset.json")
print("[1/4] Copied dataset.json")

trainer_folders = list(results_dir.glob("nnUNetTrainer__*__3d_fullres"))
if not trainer_folders:
    print("ERROR: No trained model found!")
else:
    trainer_folder  = trainer_folders[0]
    export_trainer  = export_dir / trainer_folder.name
    export_trainer.mkdir(exist_ok=True)
    for fname in ["dataset.json", "dataset_fingerprint.json", "plans.json"]:
        src = trainer_folder / fname
        if src.exists(): shutil.copy2(src, export_trainer / fname)
    for fold_dir in sorted(trainer_folder.glob("fold_*")):
        dst_fold = export_trainer / fold_dir.name
        if dst_fold.exists(): shutil.rmtree(dst_fold)
        shutil.copytree(fold_dir, dst_fold)
        print(f"  Copied {fold_dir.name}")
    print(f"[2/4] Exported trainer: {trainer_folder.name}")
    pp = results_dir / "postprocessing.pkl"
    if pp.exists():
        shutil.copy2(pp, export_dir / "postprocessing.pkl")
        print("[3/4] Copied postprocessing.pkl")
    else:
        print("[3/4] No postprocessing.pkl (run find_best_configuration first)")
    (export_dir / "README.txt").write_text(
        "nnUNetV2 Model Export — Dataset501_KidneyStones\n"
        "Config: 3d_fullres\n"
        "Folds : fold_0 to fold_4\n"
    )
    print("[4/4] Created README.txt")

print(f"\nExport: {export_dir}")

In [ ]:
import shutil
from pathlib import Path

zip_path   = Path(RESULTS_DRIVE_PATH) / "streamlit_model_export_501.zip"
export_dir = Path("/content/streamlit_export_501")

shutil.make_archive(str(zip_path).replace(".zip", ""), "zip", export_dir)
size_mb = zip_path.stat().st_size / 1e6
print(f"Zip saved: {zip_path}")
print(f"Size: {size_mb:.1f} MB")